<a href="https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/feature/marquise-transcoder-feature-inspection/notebooks/pi05_e5_dictionary_health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# E5 — Dictionary occupancy and how sparse the code really is

**Question.** The transcoders are called sparse and given a sixteen-fold
expansion, and neither the sparsity nor the occupancy of that dictionary has
been reported. How much of the 16384 is used, how much is dead, and how many
features are active at one action token?

**The catch this notebook is built around.** Counting features that never fired
does not count dead features, it bounds them. A feature seen zero times in 160
observations could still fire on 1.85% of them, which is eleven times the rate
of the paper's own case-study feature. Separating dead from rare needs about
1800 observations, so a small run must report a bound and say so rather than
quote a dead count.

**Two parts.**

* **Part A** runs the occupancy analysis on a discovery run you already have.
  No GPU, about a minute. If that run is small, it will tell you so.
* **Part B** runs a fresh discovery pass sized to make the dead count
  identifiable, and which also records per-token L0 — the statistic the word
  "sparse" actually refers to, and the one the occupancy table can only bound
  from above. Needs a GPU and roughly forty minutes.

Run Part A first. If it says the sample is too small, run Part B.


In [ ]:
# @title Controls

DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
TRANSCODER_DRIVE_PATH = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"  # @param {type:"string"}
POLICY_PATH = "lerobot/pi05_libero_finetuned"  # @param {type:"string"}

REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "feature/marquise-transcoder-feature-inspection"  # @param {type:"string"}

# Part A: leave blank to use the most recent discovery run found on Drive.
FEATURE_DIR = ""  # @param {type:"string"}

# The firing rate the run must be able to detect. The default is the rate of the
# paper's case-study feature L11:tau0.7:F9970, so a run that clears it can speak
# about the paper's own example.
REFERENCE_RATE = 0.00166  # @param {type:"number"}

# Part B: a fresh discovery pass. 750 batches x 4 = 3000 observations, which
# clears the ~1800 needed for the default reference rate with margin.
RUN_FRESH_DISCOVERY = False  # @param {type:"boolean"}
DISCOVERY_EPISODES = "0-19"  # @param {type:"string"}
DISCOVERY_MAX_BATCHES = 750  # @param {type:"integer"}
DISCOVERY_BATCH_SIZE = 4  # @param {type:"integer"}

SAVE_TO_DRIVE = True  # @param {type:"boolean"}

import math
planned = DISCOVERY_MAX_BATCHES * DISCOVERY_BATCH_SIZE
needed = math.ceil(math.log(0.05) / math.log(1 - REFERENCE_RATE))
print(f"reference rate {REFERENCE_RATE:g} needs about {needed} observations to separate dead from rare")
print(f"Part B as configured would collect {planned} observations -> "
      f"{'sufficient' if planned >= needed else 'NOT sufficient, raise DISCOVERY_MAX_BATCHES'}")


In [ ]:
# @title Mount Drive And Clone

import subprocess, time
from pathlib import Path
from google.colab import drive


def mount_with_retry(mountpoint="/content/drive", attempts=3):
    if Path(mountpoint, "MyDrive").exists():
        print("Drive already mounted"); return
    for attempt in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=attempt > 1); print(f"Drive mounted (attempt {attempt})"); return
        except Exception as exc:
            print(f"attempt {attempt}/{attempts} failed: {exc}")
            if attempt < attempts:
                time.sleep(5 * attempt)
    raise RuntimeError("Could not mount Drive. Runtime > Manage sessions, end other sessions, retry.")


mount_with_retry()
DRIVE_ROOT = Path(DRIVE_ROOT)

LOCAL_REPO = Path("/content/pi05-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)
subprocess.run(["git", "-C", str(LOCAL_REPO), "log", "-1", "--oneline"], check=True)


## Part A — Occupancy of an existing discovery run

Reads `feature_stats.pt` and reports, per layer, how much of the dictionary is
ever used and how often the used features fire. It also reports whether the run
is large enough for the never-fired count to mean anything, and if not, the
sample size that would be.


In [ ]:
# @title Run The Occupancy Analysis

import os, subprocess, sys
from pathlib import Path

if FEATURE_DIR.strip():
    feature_dir = Path(FEATURE_DIR.strip())
else:
    roots = [DRIVE_ROOT / "outputs/features", LOCAL_REPO / "outputs/features"]
    found = [p.parent for root in roots if root.exists() for p in root.rglob("feature_stats.pt")]
    if not found:
        raise FileNotFoundError("No feature_stats.pt found. Set FEATURE_DIR, or run Part B.")
    feature_dir = max(found, key=lambda p: (p / "feature_stats.pt").stat().st_mtime)
print("discovery run:", feature_dir)

out_dir = Path("/content/e5_dictionary_health") / feature_dir.name
out_dir.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
cmd = [sys.executable, "-u", "scripts/analyze_dictionary_health.py", str(feature_dir),
       "--output-dir", str(out_dir), "--reference-rate", str(REFERENCE_RATE)]
print("$", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-3000:]); raise SystemExit(f"analysis exited {proc.returncode}")


In [ ]:
# @title Table, Figure And Verdict

import json
from IPython.display import display, HTML, Markdown
import matplotlib.pyplot as plt

report = json.loads((out_dir / "dictionary_health.json").read_text())
rows, summary, ts = report["layers"], report["summary"], report.get("token_sparsity", {})

def _f(v, spec=".3f"):
    return "n/a" if v is None else format(float(v), spec)

display(Markdown(f"### Occupancy by layer ({summary['observations']} observations)"))
display(HTML(
    "<table><tr><th>layer</th><th>never fired</th><th>%</th><th>median rate</th>"
    "<th>max rate</th><th>E[active]</th><th>% of dictionary</th></tr>"
    + "".join(
        f"<tr><td>{r['layer_index']}</td><td>{r['features_never_fired']}</td>"
        f"<td>{r['never_fired_fraction']*100:.0f}%</td><td>{_f(r.get('rate_median'),'.3f')}</td>"
        f"<td>{_f(r.get('rate_max'),'.3f')}</td><td>{r['expected_active_per_observation']:.0f}</td>"
        f"<td>{100*r['expected_active_per_observation']/r['features_total']:.0f}%</td></tr>"
        for r in rows)
    + "</table>"
))

idx = [r["layer_index"] for r in rows]
fig, ax1 = plt.subplots(figsize=(8, 3.4))
ax1.bar(idx, [100 * r["never_fired_fraction"] for r in rows], color="tab:red", alpha=0.35,
        label="never fired (% of dictionary)")
ax1.set_xlabel("action-expert layer"); ax1.set_ylabel("% never fired", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(idx, [r.get("rate_median", 0) for r in rows], "o-", color="tab:blue",
         label="median firing rate of live features")
ax2.set_ylabel("median firing rate", color="tab:blue"); ax2.set_ylim(0, 1.05)
ax1.set_title("A bimodal dictionary: unused at the ends, always-on where it is used")
fig.tight_layout(); fig.savefig(out_dir / "dictionary_health.png", dpi=150); plt.show()

if ts.get("available"):
    display(Markdown("### Per-token L0 — the actual sparsity"))
    display(HTML(
        "<table><tr><th>quantity</th><th>median</th><th>mean</th><th>p90</th><th>max</th></tr>"
        f"<tr><td>L0 per token</td><td>{ts['l0_median']:.0f}</td><td>{ts['l0_mean']:.1f}</td>"
        f"<td>{ts['l0_p90']:.0f}</td><td>{ts['l0_max']:.0f}</td></tr>"
        f"<tr><td>% of {ts['d_features']}</td><td>{100*ts['l0_median']/ts['d_features']:.2f}%</td>"
        f"<td>{ts['l0_mean_pct']:.2f}%</td><td>{100*ts['l0_p90']/ts['d_features']:.2f}%</td>"
        f"<td>{100*ts['l0_max']/ts['d_features']:.2f}%</td></tr></table>"
    ))
    print(f"per-layer mean L0 ranges {ts['layer_mean_range'][0]:.1f} to {ts['layer_mean_range'][1]:.1f}"
          f" over {ts['tokens']:,} tokens")
else:
    display(Markdown(
        "### Per-token L0 — not available\n\n"
        f"<small>{ts.get('reason','')}</small>\n\n"
        "Until this is recorded, the E\\[active\\] column above is only an **upper bound** on "
        "sparsity: a feature counts there if it fired at *any* of the fifty action tokens, so the "
        "bound can be loose by up to fifty times. Set `RUN_FRESH_DISCOVERY = True` to get it."
    ))

display(Markdown(f"### Verdict\n\n**{summary['verdict']}**\n\n<small>rule: {summary['decision_rule']}</small>"))


## Part B — A discovery pass large enough to answer the question

Only needed when Part A says the sample is too small, or when `token_sparsity.pt`
is missing. This installs the runtime, downloads the LIBERO dataset and runs
feature discovery over enough observations that a never-fired feature really is
dead. It also records per-token L0, which earlier runs did not.

Roughly forty minutes on an L4 at the default settings.


In [ ]:
# @title Install Runtime (Part B only)

import os, subprocess
from pathlib import Path

if not RUN_FRESH_DISCOVERY:
    print("RUN_FRESH_DISCOVERY is off; skipping. Part A's result stands.")
else:
    VENV = Path("/content/lerobot-venv"); PYTHON = VENV / "bin/python"
    UV_BIN_DIR = Path("/content/uv-bin"); UV = str(UV_BIN_DIR / "uv")

    def run(cmd, env=None):
        cmd = list(map(str, cmd)); print("$", " ".join(cmd), flush=True)
        return subprocess.run(cmd, env=env, check=True)

    if PYTHON.exists():
        print("venv already present; skipping install.")
    else:
        apt_env = os.environ.copy(); apt_env["DEBIAN_FRONTEND"] = "noninteractive"
        run(["apt-get", "update", "-qq"], env=apt_env)
        run(["apt-get", "install", "-y", "-qq", "build-essential", "cmake", "curl", "ffmpeg",
             "git", "pkg-config", "libgl1", "libglib2.0-0"], env=apt_env)
        UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
        run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", "/tmp/install-uv.sh"])
        uv_env = os.environ.copy(); uv_env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
        run(["sh", "/tmp/install-uv.sh"], env=uv_env)
        os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]
        run([UV, "python", "install", "3.12"])
        run([UV, "venv", "--clear", str(VENV), "--python", "3.12"])
        run([UV, "pip", "install", "--python", str(PYTHON), "--torch-backend", "cu128",
             "lerobot[pi]", "hf-transfer", "numpy"])
    os.environ["PATH"] = f"{VENV / 'bin'}:{UV_BIN_DIR}:" + os.environ["PATH"]
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    run([str(PYTHON), "-c", "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"])


In [ ]:
# @title Run Fresh Discovery (Part B only)

import os, shutil, subprocess, sys, time
from pathlib import Path

if not RUN_FRESH_DISCOVERY:
    print("RUN_FRESH_DISCOVERY is off; skipping.")
else:
    token = os.environ.get("HF_TOKEN", "")
    if not token:
        try:
            from google.colab import userdata; token = userdata.get("HF_TOKEN") or ""
        except Exception:
            token = ""
    if not token and (DRIVE_ROOT / "secrets/HF_TOKEN.txt").exists():
        token = (DRIVE_ROOT / "secrets/HF_TOKEN.txt").read_text().strip()
    if not token:
        raise RuntimeError(f"No HF token. Add HF_TOKEN as a Colab secret; {POLICY_PATH} is gated.")
    os.environ["HF_TOKEN"] = token

    candidate = Path(TRANSCODER_DRIVE_PATH)
    if not candidate.is_absolute():
        candidate = DRIVE_ROOT / candidate
    if not candidate.exists():
        raise FileNotFoundError(f"Transcoder checkpoint not found: {candidate}")
    local_ckpt = Path("/content/checkpoints") / candidate.name
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    if not (local_ckpt.exists() and local_ckpt.stat().st_size == candidate.stat().st_size):
        print(f"copying {candidate.stat().st_size / 1024**3:.2f} GiB off Drive...", flush=True)
        shutil.copy2(candidate, local_ckpt)

    if "-" in DISCOVERY_EPISODES:
        lo, hi = DISCOVERY_EPISODES.split("-"); episodes = ",".join(str(i) for i in range(int(lo), int(hi) + 1))
    else:
        episodes = DISCOVERY_EPISODES

    stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
    fresh_dir = LOCAL_REPO / "outputs/features/pi05_libero" / f"dictionary-health-{stamp}"
    PYTHON = Path("/content/lerobot-venv/bin/python")
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"; env["MPLBACKEND"] = "Agg"
    env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

    cmd = [str(PYTHON), "-u", "scripts/collect_pi05_transcoder_features.py",
           "--checkpoint", str(local_ckpt), "--output-dir", str(fresh_dir),
           "--episodes", episodes, "--max-batches", str(DISCOVERY_MAX_BATCHES),
           "--batch-size", str(DISCOVERY_BATCH_SIZE),
           "--device", "cuda", "--policy-dtype", "bfloat16"]
    print("$", " ".join(cmd), flush=True)
    log = fresh_dir / "discovery.log"; fresh_dir.mkdir(parents=True, exist_ok=True)
    with log.open("wb") as handle:
        proc = subprocess.Popen(cmd, cwd=LOCAL_REPO, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
        try:
            while True:
                chunk = proc.stdout.read(4096)
                if not chunk:
                    break
                handle.write(chunk); handle.flush()
                stream = getattr(sys.stdout, "buffer", None)
                (stream.write(chunk) if stream is not None else sys.stdout.write(chunk.decode("utf-8", "replace")))
                sys.stdout.flush()
            rc = proc.wait()
        except BaseException:
            proc.kill(); proc.wait(); raise
    if rc != 0:
        raise SystemExit(f"discovery exited {rc}; see {log}")
    print("\nFresh discovery run:", fresh_dir)
    print("Set FEATURE_DIR to this path and re-run Part A, or just re-run the Part A cells "
          "(they pick the most recent run).")


In [ ]:
# @title Save To Drive

import shutil
from pathlib import Path

if SAVE_TO_DRIVE:
    target = DRIVE_ROOT / "outputs/experiments/e5_dictionary_health" / out_dir.name
    shutil.copytree(out_dir, target, dirs_exist_ok=True)
    print("analysis saved ->", target)
    if RUN_FRESH_DISCOVERY and "fresh_dir" in dir():
        rel = Path(fresh_dir).relative_to(LOCAL_REPO)
        shutil.copytree(fresh_dir, DRIVE_ROOT / rel, dirs_exist_ok=True)
        print("discovery run saved ->", DRIVE_ROOT / rel)
else:
    print("SAVE_TO_DRIVE is off; results are lost when the runtime ends.")
